<img src="https://raw.githubusercontent.com/Delarbol/semillero/refs/heads/master/Images-DS/alt_banner.png" width=100%>
<h1 align=center style="font-size:3vw">Departamento de Matemáticas</h1>
<h1 align=center style="font-size:3vw">IA para Economía II</h1>
<h2 align=center>Economía</h2>
<h2 align=center>Camilo Andrés De la Cruz Arboleda - David Díaz Guzmán</h2>



<img src="https://raw.githubusercontent.com/Delarbol/semillero/refs/heads/master/Images-DS/attrib.png">


---


# 🌱 Árboles de Decisión para Política Pública: Acceso a Microcréditos
**Tema:** Árboles de Decisión para apoyar decisiones de asignación de microcréditos.

---
## 🎯 Contexto del problema (política pública)
Un programa de reactivación económica dispone de un presupuesto limitado para **microcréditos** (o capital semilla).  
El gobierno quiere una herramienta que ayude a priorizar a pequeños emprendedores con mayor probabilidad de **éxito en el pago**.

> **Objetivo del modelo:** predecir `exito_pago` (1 = pagó exitosamente; 0 = incumplió) a partir de variables observables.

---
## ✅ Lo que haremos en este cuaderno
1. Introducción intuitiva a Árboles de Decisión, **Entropía** e **Índice de Gini**.
2. Generar un **dataset simulado** (1,000 observaciones) con variables socioeconómicas.
3. EDA (análisis exploratorio) con visualizaciones.
4. Entrenar un `DecisionTreeClassifier` en scikit-learn.
5. Visualizar el árbol (reglas de decisión).
6. Evaluación “económica”: matriz de confusión y consecuencias de errores.
7. Reflexión ética: sesgos, exclusión y gobernanza del algoritmo.



## 1) Introducción teórica (intuitiva)

### ¿Qué es un Árbol de Decisión?
Un árbol de decisión aprende **reglas tipo “si-entonces”** para clasificar o predecir.  
Ejemplo:
- Si `ahorros_previos` > 2 millones y `años_experiencia` > 3 → mayor probabilidad de `exito_pago = 1`.

En economía pública, los árboles son útiles porque:
- Capturan **no-linealidades** (umbrales: “si no supera X, cae la probabilidad”).
- Capturan **interacciones** (la educación puede importar más en ciertos sectores).
- Son relativamente **interpretables** (reglas comunicables).

### Entropía (intuición)
Mide la “incertidumbre” de una muestra respecto a la clase (éxito/fracaso):
- 50%/50% → alta incertidumbre → entropía alta
- 95%/5% → baja incertidumbre → entropía baja

El árbol busca divisiones que reduzcan esa incertidumbre.

### Índice de Gini (intuición)
También mide impureza:
- Gini = 0 → grupo puro
- Gini alto → mezcla de clases

En práctica, Gini y Entropía suelen producir árboles similares.

### ¿Por qué frente a Regresión Logística?
La regresión logística impone una forma funcional más “suave” (lineal en parámetros).  
En política pública y economía aplicada, muchos procesos son **no lineales** y **segmentados**.  
El árbol aproxima reglas por segmentos sin imponer una única forma.


## 2) Librerías y configuración

In [ ]:

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_theme()
plt.rcParams["figure.figsize"] = (8, 5)

np.random.seed(42)



## 3) Dataset simulado (1,000 observaciones)

**Variables:**
- `años_experiencia` (0 a 20)
- `ahorros_previos` (millones, 0 a 20)
- `sector_economico` (comercio, servicios, agricultura, manufactura)
- `educacion` (primaria, secundaria, tecnica, universitaria)
- `exito_pago` (0/1)

Simulamos una realidad con **no-linealidades** e **interacciones**, más ruido.


In [ ]:

n = 1000

anios_experiencia = np.random.randint(0, 21, size=n)
ahorros_previos = np.clip(np.random.normal(loc=4, scale=3, size=n), 0, 20)

sector_economico = np.random.choice(
    ["comercio", "servicios", "agricultura", "manufactura"],
    size=n,
    p=[0.35, 0.35, 0.2, 0.1]
)

educacion = np.random.choice(
    ["primaria", "secundaria", "tecnica", "universitaria"],
    size=n,
    p=[0.2, 0.45, 0.25, 0.1]
)

base = -1.0
score = base + 0.12 * anios_experiencia + 0.35 * np.log1p(ahorros_previos)

edu_effect = {"primaria": -0.25, "secundaria": 0.0, "tecnica": 0.25, "universitaria": 0.35}
score += np.vectorize(edu_effect.get)(educacion)

sector_effect = {"comercio": 0.10, "servicios": 0.15, "agricultura": -0.35, "manufactura": -0.10}
score += np.vectorize(sector_effect.get)(sector_economico)

# Interacción no lineal
score += np.where((sector_economico == "manufactura") & (ahorros_previos < 2), -0.5, 0.0)

p = 1 / (1 + np.exp(-score))
p = np.clip(p + np.random.normal(0, 0.06, size=n), 0.01, 0.99)

exito_pago = np.random.binomial(1, p, size=n)

df = pd.DataFrame({
    "años_experiencia": anios_experiencia,
    "ahorros_previos": ahorros_previos,
    "sector_economico": sector_economico,
    "educacion": educacion,
    "exito_pago": exito_pago
})

df.head()


### Distribución de la variable objetivo

In [ ]:

df["exito_pago"].value_counts(normalize=True).rename("proporción")



## 4) EDA (Análisis Exploratorio)

Vamos a inspeccionar patrones iniciales:
- variables numéricas vs éxito
- variables categóricas vs éxito
- correlaciones numéricas


### 4.1 Numéricas vs objetivo

In [ ]:

sns.boxplot(data=df, x="exito_pago", y="años_experiencia")
plt.title("Años de experiencia según éxito de pago")
plt.show()

sns.boxplot(data=df, x="exito_pago", y="ahorros_previos")
plt.title("Ahorros previos (millones) según éxito de pago")
plt.show()


### 4.2 Categóricas vs objetivo (tasa de éxito)

In [ ]:

sector_rate = df.groupby("sector_economico")["exito_pago"].mean().sort_values(ascending=False)
sns.barplot(x=sector_rate.index, y=sector_rate.values)
plt.title("Tasa promedio de éxito de pago por sector")
plt.ylabel("Tasa de éxito")
plt.xlabel("Sector económico")
plt.show()

edu_rate = df.groupby("educacion")["exito_pago"].mean().sort_values(ascending=False)
sns.barplot(x=edu_rate.index, y=edu_rate.values)
plt.title("Tasa promedio de éxito de pago por educación")
plt.ylabel("Tasa de éxito")
plt.xlabel("Nivel educativo")
plt.show()


### 4.3 Correlaciones (solo numéricas)

In [ ]:

corr = df[["años_experiencia", "ahorros_previos", "exito_pago"]].corr()
sns.heatmap(corr, annot=True)
plt.title("Correlación (variables numéricas)")
plt.show()



## 5) Modelado – DecisionTreeClassifier

### Preprocesamiento
Convertimos variables categóricas a dummies (One-Hot Encoding).


In [ ]:

X = df.drop(columns=["exito_pago"])
y = df["exito_pago"]

categorical_cols = ["sector_economico", "educacion"]
numeric_cols = ["años_experiencia", "ahorros_previos"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_test.shape



### Entrenamiento del árbol
Controlamos complejidad para evitar overfitting y mantener interpretabilidad.


In [ ]:

tree_clf = DecisionTreeClassifier(
    criterion="gini",     # pruebe también "entropy"
    max_depth=4,
    min_samples_leaf=20,
    random_state=42
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("tree", tree_clf)
])

model.fit(X_train, y_train)
print("✅ Modelo entrenado.")


## 6) Evaluación (más allá del accuracy)

In [ ]:

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))



### Matriz de confusión – lectura económica (política pública)

- **FP (Falso Positivo):** asigno crédito a quien probablemente no pagará → **pérdida / mala asignación de fondos públicos**.
- **FN (Falso Negativo):** rechazo a quien sí habría pagado → **exclusión financiera y oportunidad perdida**.

La política debe decidir qué error es más costoso.


In [ ]:

cm = confusion_matrix(y_test, y_pred)
cm


In [ ]:

sns.heatmap(cm, annot=True, fmt="d")
plt.title("Matriz de confusión (test)")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()



## 7) Visualización del árbol (plot_tree)
Veremos reglas tipo “si-entonces”.


In [ ]:

ohe = model.named_steps["preprocess"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
feature_names = list(cat_feature_names) + numeric_cols

tree_est = model.named_steps["tree"]

plt.figure(figsize=(18, 8))
plot_tree(
    tree_est,
    feature_names=feature_names,
    class_names=["fracaso (0)", "exito (1)"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Árbol de decisión para éxito de pago (microcréditos)")
plt.show()



## 8) Mini-lab: complejidad vs interpretabilidad
Probamos distintos `max_depth` y comparamos accuracy.


In [ ]:

depths = [2, 3, 4, 5, 8]
results = []

for d in depths:
    tree = DecisionTreeClassifier(
        criterion="gini",
        max_depth=d,
        min_samples_leaf=20,
        random_state=42
    )
    m = Pipeline(steps=[("preprocess", preprocess), ("tree", tree)])
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    results.append({"max_depth": d, "accuracy": accuracy_score(y_test, pred)})

pd.DataFrame(results)



**Pregunta:** ¿En qué punto el aumento de precisión deja de justificar perder interpretabilidad?  
(Escriba 4–8 líneas)


**TU RESPUESTA:**

- 



## 9) Conclusión: ética algorítmica y sesgo

Un modelo puede amplificar desigualdades si aprende proxies (p.ej. educación como proxy de ingreso),
o si se usa sin transparencia y sin mecanismos de apelación.

Preguntas clave:
1) ¿Cómo explicar una decisión automatizada a un ciudadano?  
2) ¿Qué variables deberían *no* usarse por riesgo de discriminación?  
3) ¿Quién define el umbral de aprobación? ¿con qué legitimidad?  



### ✅ Cierre (entrega)
Responda (máx. 10 líneas):
1) ¿Qué variable fue más importante según el árbol?  
2) ¿Qué error es más grave: FP o FN? (depende del objetivo)  
3) ¿Qué riesgos éticos ve en automatizar esta decisión?  


**TU RESPUESTA:**

1) 
2) 
3) 
